In [4]:
import csv

url_file = "/home/rishi/ML Projects/Air Pollution/ParquetFilesUrls (1).csv"
output_file = "/home/rishi/ML Projects/Air Pollution/parquet_urls_1.txt"

with open(url_file, encoding="utf-8-sig") as f_in, open(output_file, "w") as f_out:
    reader = csv.DictReader(f_in)
    for row in reader:
        f_out.write(row["ParquetFileUrl"] + "\n")

In [2]:
import os
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

download_folders = {
    "e1a": "/home/rishi/ML Projects/Air Pollution/eea/urlso",
    "e2a": "/home/rishi/ML Projects/Air Pollution/eea/urls1",
}
base_out_dir = "/home/rishi/ML Projects/Air Pollution/EEA"
start_date   = pd.Timestamp("2022-01-01")
end_date     = pd.Timestamp("2025-12-31 23:59:59")
COUNTRIES    = {"FR", "DE"}
MAX_WORKERS  = 24

for c in COUNTRIES:
    os.makedirs(os.path.join(base_out_dir, c), exist_ok=True)

def process(fname, folder, suffix):
    fpath = os.path.join(folder, fname)
    df = pd.read_parquet(fpath)
    if df.empty:
        return "skip"
    country = df["Samplingpoint"].iloc[0].split("/")[0]
    if country not in COUNTRIES:
        return "skip"
    df = df[(df["Start"] >= start_date) & (df["Start"] <= end_date)]
    if df.empty:
        return "skip"
    stem = fname.replace(".parquet", "")
    out_path = os.path.join(base_out_dir, country, f"{stem}_{suffix}.csv")
    df.to_csv(out_path, index=False)
    return "save"

saved, skipped = 0, 0
for suffix, folder in download_folders.items():
    parquet_files = [f for f in os.listdir(folder) if f.endswith(".parquet")]
    print(f"{suffix}: {len(parquet_files)} parquet files")
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(process, f, folder, suffix): f for f in parquet_files}
        for future in tqdm(as_completed(futures), total=len(futures), desc=suffix):
            try:
                if future.result() == "save":
                    saved += 1
                else:
                    skipped += 1
            except Exception as e:
                print(f"Error processing {futures[future]}: {e}")
                skipped += 1

print(f"Saved {saved}, skipped {skipped}")

e1a: 3925 parquet files


e1a: 100%|██████████| 3925/3925 [05:53<00:00, 11.11it/s]


e2a: 2952 parquet files


e2a: 100%|██████████| 2952/2952 [01:42<00:00, 28.69it/s]

Saved 5881, skipped 996


In [3]:
import os
import pandas as pd
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

input_base  = "/home/rishi/ML Projects/Air Pollution/EEA"
start_date  = pd.Timestamp("2022-07-01")
end_date    = pd.Timestamp("2025-06-30 23:00:00")
COUNTRIES   = ["FR", "DE"]
MAX_WORKERS = 24

EEA_POL_MAP = {
    1:    ("SO2 (µg/m³)",   "SO2"),
    5:    ("PM10 (µg/m³)",  "PM10"),
    7:    ("Ozone (µg/m³)", "Ozone"),
    8:    ("NO2 (µg/m³)",   "NO2"),
    10:   ("CO (mg/m³)",    "CO"),
    6001: ("PM2.5 (µg/m³)", "PM2.5"),
}
EXPECTED_UNITS = {
    1: "ug.m-3", 5: "ug.m-3", 7: "ug.m-3",
    8: "ug.m-3", 10: "mg.m-3", 6001: "ug.m-3",
}

full_index = pd.date_range(start=start_date, end=end_date, freq="h", name="Timestamp")

def extract_site_id(samplingpoint):
    part = samplingpoint.split("/")[1]
    if part.startswith("SPO.DE_"):
        return part[len("SPO.DE_"):].split("_")[0]
    elif part.startswith("SPO-"):
        return part[len("SPO-"):].rsplit("_", 1)[0]
    return part

def process_group(paths, proc_dir):
    dfs = []
    for p in paths:
        df = pd.read_csv(p, parse_dates=["Start"])
        df = df[df["Validity"] >= 1]
        dfs.append(df)

    combined = pd.concat(dfs, ignore_index=True)
    if combined.empty:
        return "skip"

    pol_id = int(combined["Pollutant"].iloc[0])
    if pol_id not in EEA_POL_MAP:
        return "skip"
    col_name, formula = EEA_POL_MAP[pol_id]

    unit = str(combined["Unit"].iloc[0])
    if unit != EXPECTED_UNITS[pol_id]:
        print(f"WARNING: unexpected unit '{unit}' for pollutant {pol_id}")

    site_id = extract_site_id(combined["Samplingpoint"].iloc[0])

    combined["Value"] = pd.to_numeric(combined["Value"], errors="coerce")
    out = (combined[["Start", "Value"]]
           .rename(columns={"Start": "Timestamp", "Value": col_name})
           .set_index("Timestamp")
           .sort_index())
    out = out[~out.index.duplicated(keep="first")]
    out = out.reindex(full_index)

    out.to_csv(os.path.join(proc_dir, f"site_{site_id}_{formula}.csv"))
    return "save"

saved, skipped = 0, 0
for country in COUNTRIES:
    input_dir = os.path.join(input_base, country)
    proc_dir  = os.path.join(input_base, country, "processed")
    os.makedirs(proc_dir, exist_ok=True)

    # Group files by (site_id, pollutant_id) before processing
    groups = defaultdict(list)
    for fname in os.listdir(input_dir):
        if not fname.endswith(".csv"):
            continue
        fpath = os.path.join(input_dir, fname)
        # peek at just the key columns to get the group key
        peek = pd.read_csv(fpath, usecols=["Samplingpoint", "Pollutant"], nrows=1)
        if peek.empty:
            continue
        site_id = extract_site_id(peek["Samplingpoint"].iloc[0])
        pol_id  = int(peek["Pollutant"].iloc[0])
        groups[(site_id, pol_id)].append(fpath)

    print(f"{country}: {len(os.listdir(input_dir))} files → {len(groups)} site+pollutant groups")

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {
            executor.submit(process_group, paths, proc_dir): key
            for key, paths in groups.items()
        }
        for future in tqdm(as_completed(futures), total=len(futures), desc=country):
            try:
                if future.result() == "save":
                    saved += 1
                else:
                    skipped += 1
            except Exception as e:
                print(f"Error on {futures[future]}: {e}")
                skipped += 1

print(f"Done. Saved {saved}, skipped {skipped}")

FR: 2779 files → 1542 site+pollutant groups


FR: 100%|██████████| 1542/1542 [01:50<00:00, 13.91it/s]


DE: 3104 files → 1649 site+pollutant groups


DE: 100%|██████████| 1649/1649 [02:00<00:00, 13.68it/s]


Done. Saved 3187, skipped 4
